# The command line interface

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

Larvaworld can be driven entirely from a terminal. The `larvaworld` command runs any of the
platform's simulation modes without you writing a single line of Python, which makes it the
quickest way to reproduce a run, to script a series of experiments, or to check that your
installation works.

This notebook is a tour of that command. It calls the real executable with `!`, so everything
you see below is exactly what you would get in your own shell.

**What you will be able to do afterwards**

- Run any of the five simulation modes from a terminal.
- Read `larvaworld -h` and the per-mode help without guessing what an argument does.
- Know why argument order matters, and how to avoid the one mistake everybody makes.
- Find, from Python, the complete list of arguments a mode accepts.

**Prerequisites** : Larvaworld installed (`pip install larvaworld`).

**Cost** : seconds. One cell launches a deliberately tiny simulation - two larvae for three
simulated seconds - because the flag it demonstrates cannot be shown without one.

## Setup

Nothing here is needed for the `larvaworld` command itself, which is a standalone executable. The
imports are for the last section, where we look at how the command builds its arguments.

`PYTHONWARNINGS` is set only to keep third-party deprecation warnings out of the output below; it
is not something you need in your own shell.

In [1]:
%env PYTHONWARNINGS=ignore

import warnings

# ScreenOps pulls in pygame, which emits a third-party deprecation notice on
# import. It says nothing about larvaworld, so it is silenced here.
warnings.filterwarnings("ignore", category=UserWarning)

import larvaworld
from larvaworld.cli.argparser import ParserArgumentDict, SimModeParser
from larvaworld.lib import reg
from larvaworld.lib.param import SimOps
from larvaworld.lib.screen import ScreenOps

print(f"larvaworld {larvaworld.__version__}")

env: PYTHONWARNINGS=ignore


Initializing larvaworld registry


Registry configured!


larvaworld 2.4.0


## Section 1 : The `larvaworld` command

The package installs one console entry point, `larvaworld`. Called with `--help` (or `-h`) it
prints the general help : the global options, and the list of simulation modes it accepts.

In [2]:
# The general help message
!larvaworld -h

Initializing larvaworld registry
Registry configured!
usage: larvaworld [-h] [--version] [-verbose VERBOSE] [-parsargs]
                  {Exp,Batch,Ga,Eval,Replay} ...

CLI for running larvaworld simulations

positional arguments:
  {Exp,Batch,Ga,Eval,Replay}
                        The simulation mode to launch

options:
  -h, --help            show this help message and exit
  --version             Show the version number and exit
  -verbose VERBOSE, --VERBOSE VERBOSE
                        Level of verbosity in the output
  -parsargs, --show_parser_args
                        Whether to show the parser argument namespace


Two global flags are worth knowing straight away.

**`-verbose LEVEL`** controls how much the *run* reports about what it is doing. Note that the two
registry lines above it are printed while the package is imported, before the flag can take effect,
so they appear at every verbosity level.

**`-parsargs`** prints the parsed argument namespace, which is the fastest way to check what your
command line actually means before trusting it. It prints and then continues into the run, so the
example below is given a deliberately tiny experiment.

In [3]:
# Verbosity applies to the run; -h exits before anything is launched
!larvaworld -verbose 0 Exp -h

Initializing larvaworld registry
Registry configured!
usage: larvaworld Exp [-h] [-Box2D] [-larva_collisions] [-fr FR] [-dt DT]
                      [-constant_framerate] [-duration DURATION]
                      [-Nsteps NSTEPS] [-N NAGENTS] [-a] [-show]
                      [-image_mode {final,snapshots,overlap}]
                      [-image_file IMAGE_FILE]
                      [-snapshot_interval_in_sec SNAPSHOT_INTERVAL_IN_SEC]
                      [-video_file VIDEO_FILE] [-media_dir MEDIA_DIR]
                      [-fps FPS] [-save_video] [-vis_mode {video,image}]
                      [-show_display] [-pygame_keys PYGAME_KEYS]
                      [-display_every_n_steps DISPLAY_EVERY_N_STEPS]
                      [-visible_trails] [-trail_dt TRAIL_DT]
                      [-trail_color {normal,linear,angular}] [-draw_sensors]
                      [-draw_contour] [-draw_segs] [-draw_midline]
                      [-draw_centroid] [-draw_head] [-draw_orientations]
   

In [4]:
# Show the parsed argument namespace. This also launches the (very short) run.
!larvaworld -parsargs Exp dish -N 2 -duration 0.05

Initializing larvaworld registry
Registry configured!
Simulation mode : Exp
Simulation args as nested dictionary: 
     screen_kws : 
          image_mode : None
          image_file : None
          snapshot_interval_in_sec : 60
          video_file : None
          media_dir : None
          fps : 60
          save_video : False
          vis_mode : None
          show_display : False
          pygame_keys : None
          display_every_n_steps : 1
          visible_trails : False
          trail_dt : 20
          trail_color : normal
          draw_sensors : False
          draw_contour : True
          draw_segs : True
          draw_midline : True
          draw_centroid : False
          draw_head : False
          draw_orientations : False
          intro_text : True
          odor_aura : False
          allow_clicks : True
          black_background : False
          random_colors : False
          color_behavior : False
          panel_width : 0
     id : None
     dir : None


## Section 2 : Choosing a simulation mode

`larvaworld` takes exactly one **positional** argument : the simulation mode. Everything else is
optional and depends on the mode you picked. Leaving the mode out is an error - the two flags above
were given one for exactly that reason.

| mode | what it does |
|---|---|
| `Exp` | a single virtual experiment - the default way to run a model |
| `Batch` | a parameter sweep, running many experiments in a defined space |
| `Ga` | genetic-algorithm optimization of a model against a reference dataset |
| `Eval` | evaluation of one or more models against a reference dataset |
| `Replay` | replay of a recorded dataset, driving the agents from the data |

> **Argument order matters.** General arguments come **before** the mode, mode-specific arguments
> **after** it :
>
> ```bash
> larvaworld -verbose 1 Exp dish -N 5      # correct
> larvaworld Exp dish -verbose 1 -N 5      # fails
> ```
>
> This is a consequence of how `argparse` sub-parsers work, not a Larvaworld convention.

The per-mode help shows what each mode adds. Compare a plain experiment with the genetic
algorithm : the second one gains a whole block of selection and evaluation options.

In [5]:
# Arguments available for a single experiment
!larvaworld Exp -h

Initializing larvaworld registry
Registry configured!
usage: larvaworld Exp [-h] [-Box2D] [-larva_collisions] [-fr FR] [-dt DT]
                      [-constant_framerate] [-duration DURATION]
                      [-Nsteps NSTEPS] [-N NAGENTS] [-a] [-show]
                      [-image_mode {final,snapshots,overlap}]
                      [-image_file IMAGE_FILE]
                      [-snapshot_interval_in_sec SNAPSHOT_INTERVAL_IN_SEC]
                      [-video_file VIDEO_FILE] [-media_dir MEDIA_DIR]
                      [-fps FPS] [-save_video] [-vis_mode {video,image}]
                      [-show_display] [-pygame_keys PYGAME_KEYS]
                      [-display_every_n_steps DISPLAY_EVERY_N_STEPS]
                      [-visible_trails] [-trail_dt TRAIL_DT]
                      [-trail_color {normal,linear,angular}] [-draw_sensors]
                      [-draw_contour] [-draw_segs] [-draw_midline]
                      [-draw_centroid] [-draw_head] [-draw_orientations]
   

In [6]:
# ... and for the genetic algorithm, which adds selection and evaluation options
!larvaworld Ga -h

Initializing larvaworld registry
Registry configured!
usage: larvaworld Ga [-h] [-Box2D] [-larva_collisions] [-fr FR] [-dt DT]
                     [-constant_framerate] [-duration DURATION]
                     [-Nsteps NSTEPS] [-image_mode {final,snapshots,overlap}]
                     [-image_file IMAGE_FILE]
                     [-snapshot_interval_in_sec SNAPSHOT_INTERVAL_IN_SEC]
                     [-video_file VIDEO_FILE] [-media_dir MEDIA_DIR]
                     [-fps FPS] [-save_video] [-vis_mode {video,image}]
                     [-show_display] [-pygame_keys PYGAME_KEYS]
                     [-display_every_n_steps DISPLAY_EVERY_N_STEPS]
                     [-visible_trails] [-trail_dt TRAIL_DT]
                     [-trail_color {normal,linear,angular}] [-draw_sensors]
                     [-draw_contour] [-draw_segs] [-draw_midline]
                     [-draw_centroid] [-draw_head] [-draw_orientations]
                     [-intro_text] [-odor_aura] [-allow_clicks]


## Section 3 : Where the arguments come from

The help text above is not written by hand. Every argument is derived automatically from the
`param.Parameterized` configuration classes that the simulation itself uses, so the CLI can never
drift away from the API : if a parameter exists in the configuration class, it is available on the
command line, with the same name, type, default and documentation.

The class that performs that translation is `ParserArgumentDict`. Handing it a configuration class
returns the arguments that class contributes - which is a convenient way to answer *"what can I
actually set for this mode?"* without reading the help text.

### Arguments contributed by the general simulation options

`SimOps` holds what every mode needs : duration, timestep, population size, and so on.

In [7]:
sim_kws = ParserArgumentDict.from_param(d0=SimOps)
sim_kws.parsargs.keylist.sorted

['Box2D',
 'Nsteps',
 'constant_framerate',
 'dt',
 'duration',
 'fr',
 'larva_collisions']

### Arguments controlling the display

`ScreenOps` holds everything about rendering : whether a window opens, whether a video is written,
the frame rate, which parts of the larva body are drawn.

In [8]:
screen_kws = ParserArgumentDict.from_param(d0=ScreenOps)
screen_kws.parsargs.keylist.sorted

['allow_clicks',
 'black_background',
 'color_behavior',
 'display_every_n_steps',
 'draw_centroid',
 'draw_contour',
 'draw_head',
 'draw_midline',
 'draw_orientations',
 'draw_segs',
 'draw_sensors',
 'fps',
 'image_file',
 'image_mode',
 'intro_text',
 'media_dir',
 'odor_aura',
 'panel_width',
 'pygame_keys',
 'random_colors',
 'save_video',
 'show_display',
 'snapshot_interval_in_sec',
 'trail_color',
 'trail_dt',
 'video_file',
 'vis_mode',
 'visible_trails']

### Arguments specific to the genetic algorithm

The GA is configured by two classes : `GAselector` decides how each generation is built from the
previous one, and `GAevaluation` decides how a genome is scored against the reference data.

In [9]:
GAselector = ParserArgumentDict.from_param(d0=reg.gen.GAselector)
GAevaluation = ParserArgumentDict.from_param(d0=reg.gen.GAevaluation)

print("selection :", GAselector.parsargs.keylist.sorted)
print()
print("evaluation :", GAevaluation.parsargs.keylist.sorted)

selection : ['Cmutation', 'Nagents', 'Nelits', 'Ngenerations', 'Pmutation', 'base_model', 'bestConfID', 'include_effector_params', 'init_mode', 'selection_ratio', 'space_mkeys', 'space_pkeys']

evaluation : ['cycle_curve_metrics', 'eval_metrics', 'exclude_func_name', 'exclusion_mode', 'fit_kws', 'fitness_func_name', 'refDir', 'refID']


### Arguments specific to replay

Replay has its own set : which dataset, which agents, which time range, how the body is
reconstructed, whether trajectories are transposed to a common origin.

In [10]:
Replay = ParserArgumentDict.from_param(d0=reg.gen.Replay)
Replay.parsargs.keylist.sorted

['agent_ids',
 'close_view',
 'draw_Nsegs',
 'env_params',
 'fix_point',
 'fix_segment',
 'overlap_mode',
 'refDir',
 'refID',
 'time_range',
 'track_point',
 'transposition']

### The parser as a whole

`SimModeParser` is what the `larvaworld` command actually instantiates. Its `parser_dicts`
attribute is the full picture : one entry per group of arguments, assembled into the sub-parser of
whichever mode you asked for.

In [11]:
P = SimModeParser()
print(P.parser_dicts.keylist)

['screen_kws', 'SimOps', 'RuntimeOps', 'Replay', 'Eval', 'GAselector', 'GAevaluation']


## Where to go next

- [Your first simulation](single_simulation.ipynb) - the same runs, from Python, with control over
  what happens afterwards.
- [The Python API](python_api_basics.ipynb) - building a configuration instead of passing flags.
- The [Basic Usage](../../usage.md) reference page lists ready-made command lines for common tasks.